# 🩸 Blood Cell Classification Using PyTorch And ResNet18

This Project Uses A ResNet18 Deep Learning Model To Classify Human Blood Cells Into Four Categories:

- Eosinophil
- Lymphocyte
- Monocyte
- Neutrophil

Dataset:
https://www.kaggle.com/datasets/paultimothymooney/blood-cells

The Dataset Is Downloaded Using KaggleHub.

Default Training Uses Five Epochs To Keep Training Accessible For CPU And Google Colab Users.

Increasing The Number Of Epochs May Improve Accuracy.

Developed For AltayBioAI.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18
import matplotlib.pyplot as plt
import os

In [ ]:
train_path = "dataset2-master/dataset2-master/images/TRAIN"
test_path = "dataset2-master/dataset2-master/images/TEST"

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.5,0.5,0.5],
        [0.5,0.5,0.5]
    )
])

In [ ]:
train_dataset = ImageFolder(
    train_path,
    transform=transform
)

test_dataset = ImageFolder(
    test_path,
    transform=transform
)

print(train_dataset.classes)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
print("Training Images:", len(train_dataset))
print("Test Images:", len(test_dataset))
print("Classes:", train_dataset.classes)

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(12,8))

for i in range(8):
    plt.subplot(2,4,i+1)
    img = images[i].permute(1,2,0)
    img = img * 0.5 + 0.5
    plt.imshow(img)
    plt.title(
        train_dataset.classes[
            labels[i]
        ]
    )
    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = resnet18(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    4
)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [ ]:
for epoch in range(5):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(
            outputs,
            labels
        )
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = (
        running_loss /
        len(train_loader)
    )

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"- Loss: {epoch_loss:.4f}"
    )

In [ ]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(
            outputs,
            1
        )
        total += labels.size(0)
        correct += (
            predicted == labels
        ).sum().item()

accuracy = 100 * correct / total

print(
    f"Accuracy: {accuracy:.2f}%"
)

In [ ]:
images, labels = next(iter(test_loader))
images_device = images.to(device)

model.eval()

with torch.no_grad():
    outputs = model(images_device)
    _, predictions = torch.max(
        outputs,
        1
    )

In [ ]:
plt.figure(figsize=(12,8))

for i in range(8):
    plt.subplot(2,4,i+1)
    img = images[i].permute(1,2,0)
    img = img * 0.5 + 0.5
    plt.imshow(img)
    true_label = train_dataset.classes[
        labels[i]
    ]
    pred_label = train_dataset.classes[
        predictions[i].cpu()
    ]
    plt.title(
        f"T: {true_label}\nP: {pred_label}"
    )
    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
torch.save(
    model.state_dict(),
    "blood_cell_resnet18.pth"
)

print("Model Saved Successfully!")

In [ ]:
print("Training Completed Successfully.")
print(f"Final Accuracy: {accuracy:.2f}%")